# Polymarket EPL Data — v4
Haalt snapshots en tijdreeksen op voor alle EPL wedstrijden vanaf 2024/2025.
Fix: home/away labeling via teamnaam matching i.p.v. volgorde.

In [2]:
import os
os.chdir('C:/Users/semwi/FPL-Core-Insights/data')

import pandas as pd
import requests
import time
from datetime import timedelta

GAMMA_API = 'https://gamma-api.polymarket.com'
CLOB_API  = 'https://clob.polymarket.com'

SNAPSHOTS_PAD = 'polymarket_snapshots.csv'
TIJDREEKS_PAD = 'polymarket_tijdreeks.csv'
FOUTEN_PAD    = 'polymarket_fouten.csv'

print('Libraries geladen')

Libraries geladen


In [7]:
# Wedstrijden laden
df_matches = pd.read_csv('match_data.csv')
df_matches['timestamp'] = pd.to_datetime(df_matches['timestamp'])

df_matches = df_matches[
    df_matches['season'].isin(['2024-2025', '2025-2026']) &
    df_matches['status'].isin(['Ended', 'finished'])
].copy().reset_index(drop=True)

print(f'{len(df_matches)} gespeelde wedstrijden')
print(df_matches['season'].value_counts())

1061 gespeelde wedstrijden
season
23/24        380
2024-2025    380
2025-2026    301
Name: count, dtype: int64


In [8]:
TEAM_KORT = {
    'Arsenal':'ars','Aston Villa':'ast','Bournemouth':'bou','Brentford':'bre',
    'Brighton & Hove Albion':'bri','Burnley':'bur','Chelsea':'che',
    'Crystal Palace':'cry','Everton':'eve','Fulham':'ful','Ipswich Town':'ips',
    'Leeds United':'lee','Leicester City':'lei','Liverpool':'liv',
    'Manchester City':'mac','Manchester United':'mun','Newcastle United':'new',
    'Nottingham Forest':'not','Southampton':'sou','Sunderland':'sun',
    'Tottenham Hotspur':'tot','West Ham United':'wes','Wolverhampton':'wol', 'Luton Town': 'lut',
'Sheffield United': 'she',
}
TEAM_LANG = {
    'Arsenal':'arsenal','Aston Villa':'aston-villa','Bournemouth':'bournemouth',
    'Brentford':'brentford','Brighton & Hove Albion':'brighton','Burnley':'burnley',
    'Chelsea':'chelsea','Crystal Palace':'crystal-palace','Everton':'everton',
    'Fulham':'fulham','Ipswich Town':'ipswich-town','Leeds United':'leeds-united',
    'Leicester City':'leicester-city','Liverpool':'liverpool',
    'Manchester City':'manchester-city','Manchester United':'manchester-united',
    'Newcastle United':'newcastle','Nottingham Forest':'nottingham-forest',
    'Southampton':'southampton','Sunderland':'sunderland',
    'Tottenham Hotspur':'tottenham','West Ham United':'west-ham',
    'Wolverhampton':'wolverhampton', 'Luton Town': 'luton-town',
'Sheffield United': 'sheffield-united',
}

# Hoe Polymarket de teamnaam schrijft in de question tekst
# Meerdere varianten per team zodat beide formats (oud/nieuw) matchen
TEAM_POLY = {
    'Arsenal':                ['arsenal'],
    'Aston Villa':            ['aston villa', 'villa'],
    'Bournemouth':            ['bournemouth'],
    'Brentford':              ['brentford'],
    'Brighton & Hove Albion': ['brighton', 'brighton & hove albion'],
    'Burnley':                ['burnley fc', 'burnley'],
    'Chelsea':                ['chelsea'],
    'Crystal Palace':         ['crystal palace', 'palace'],
    'Everton':                ['everton'],
    'Fulham':                 ['fulham'],
    'Ipswich Town':           ['ipswich town', 'ipswich'],
    'Leeds United':           ['leeds united', 'leeds'],
    'Leicester City':         ['leicester city', 'leicester'],
    'Liverpool':              ['liverpool'],
    'Manchester City':        ['manchester city', 'man city'],
    'Manchester United':      ['manchester united', 'man united', 'man utd'],
    'Newcastle United':       ['newcastle united', 'newcastle'],
    'Nottingham Forest':      ['nottingham forest', 'nottingham', 'nott forest'],
    'Southampton':            ['southampton', 'saints'],
    'Sunderland':             ['sunderland afc', 'sunderland'],
    'Tottenham Hotspur':      ['tottenham hotspur', 'tottenham', 'spurs'],
    'West Ham United':        ['west ham united', 'west ham'],
    'Wolverhampton':          ['wolverhampton wanderers', 'wolverhampton', 'wolves'],
    'Luton Town':             ['luton town', 'luton'],
    'Sheffield United':       ['sheffield united', 'sheffield utd', 'sheff utd'],
}

def slug_kandidaten(home, away, datum):
    d   = datum.strftime('%Y-%m-%d')
    h_k = TEAM_KORT.get(home, home.lower()[:3])
    a_k = TEAM_KORT.get(away, away.lower()[:3])
    h_l = TEAM_LANG.get(home, home.lower().replace(' ','-'))
    a_l = TEAM_LANG.get(away, away.lower().replace(' ','-'))
    return [
        f'epl-{h_k}-{a_k}-{d}',
        f'epl-{h_l}-vs-{a_l}-{d}',
        f'epl-{h_l}-vs-{a_l}',
    ]

def zoek_event(home, away, kickoff):
    for slug in slug_kandidaten(home, away, kickoff):
        r = requests.get(f'{GAMMA_API}/events', params={'slug': slug}, timeout=10)
        r.raise_for_status()
        data = r.json()
        if data:
            return slug, data[0]
        time.sleep(0.1)
    raise ValueError(f'Geen event: {home} vs {away} ({kickoff.date()})')


def vind_eerste_positie(question, namen):
    """Geeft de vroegste positie terug waarop één van de namen voorkomt in question."""
    posities = [question.find(naam) for naam in namen if question.find(naam) != -1]
    return min(posities) if posities else -1

def get_tokens(event_data, home, away):
    tokens = {}
    h_namen = TEAM_POLY.get(home, [home.lower()])
    a_namen = TEAM_POLY.get(away, [away.lower()])

    for market in event_data.get('markets', []):
        question = market.get('question', '').lower()
        raw_ids  = market.get('clobTokenIds', '')
        ids      = [t.strip().strip('"') for t in raw_ids.strip('[]').split(',')]
        if not ids or not ids[0]:
            continue
        token_id = ids[0]

        if 'draw' in question:
            tokens['draw'] = token_id
        elif 'win' in question or 'beat' in question:
            h_pos = vind_eerste_positie(question, h_namen)
            a_pos = vind_eerste_positie(question, a_namen)
            if h_pos != -1 and (a_pos == -1 or h_pos < a_pos):
                tokens['home'] = token_id
            elif a_pos != -1 and (h_pos == -1 or a_pos < h_pos):
                tokens['away'] = token_id

    return tokens


def get_tijdreeks(token_id, kickoff, dagen=5):
    start_ts = int((kickoff - timedelta(days=dagen)).timestamp())
    end_ts   = int(kickoff.timestamp())
    r = requests.get(f'{CLOB_API}/prices-history', params={
        'market': token_id, 'interval': 'max',
        'fidelity': 720, 'startTs': start_ts, 'endTs': end_ts,
    }, timeout=10)
    r.raise_for_status()
    history = r.json().get('history', [])
    if not history:
        return pd.DataFrame()
    df = pd.DataFrame(history)
    df['datetime'] = pd.to_datetime(df['t'], unit='s')
    df['price']    = df['p'].astype(float)
    df = df[df['datetime'] <= kickoff].sort_values('datetime', ascending=False)
    if df.empty:
        return pd.DataFrame()
    snap_tijden = [kickoff - timedelta(hours=12*i) for i in range(dagen*2)]
    resultaat = []
    for snap_tijd in snap_tijden:
        kand = df[df['datetime'] <= snap_tijd]
        if kand.empty:
            continue
        rij = kand.iloc[0].copy()
        rij['snap_tijd']         = snap_tijd
        rij['uren_voor_kickoff'] = (kickoff - snap_tijd).total_seconds() / 3600
        resultaat.append(rij)
    if not resultaat:
        return pd.DataFrame()
    return pd.DataFrame(resultaat)[['snap_tijd','uren_voor_kickoff','datetime','price']]


# ── SNELLE TEST ──────────────────────────────────────────────────────────────
test = df_matches.iloc[0]
print(f'Test: {test["home_team"]} vs {test["away_team"]} ({test["timestamp"].date()})')
try:
    slug, event = zoek_event(test['home_team'], test['away_team'], test['timestamp'])
    tokens = get_tokens(event, test['home_team'], test['away_team'])
    print(f'Slug gevonden: {slug}')
    print(f'Tokens: {list(tokens.keys())}')
    assert sorted(tokens.keys()) == ['away','draw','home'], 'Labels niet compleet!'
    print('✅ Test geslaagd')
except Exception as e:
    print(f'❌ {e}')

Test: Burnley vs Manchester City (2023-08-11)
❌ Geen event: Burnley vs Manchester City (2023-08-11)


In [9]:
# ── HOOFDLUS ─────────────────────────────────────────────────────────────────
# Laad bestaande data zodat we niet dubbel ophalen
try:
    df_snap_oud = pd.read_csv(SNAPSHOTS_PAD)
    al_opgehaald = set(df_snap_oud['match_id'])
    print(f'Bestaande snapshots: {len(df_snap_oud)} — sla deze over')
except FileNotFoundError:
    df_snap_oud = pd.DataFrame()
    al_opgehaald = set()
    print('Geen bestaand bestand — start vers')

try:
    df_tr_oud = pd.read_csv(TIJDREEKS_PAD)
except FileNotFoundError:
    df_tr_oud = pd.DataFrame()

# Filter al opgehaalde wedstrijden
te_ophalen = df_matches[~df_matches['match_id'].isin(al_opgehaald)].reset_index(drop=True)
print(f'Te ophalen: {len(te_ophalen)} wedstrijden')

snapshots   = []
tijdreeksen = []
fouten      = []
totaal      = len(te_ophalen)

for i, row in te_ophalen.iterrows():
    home     = row['home_team']
    away     = row['away_team']
    kickoff  = row['timestamp']
    match_id = row['match_id']

    print(f'[{i+1}/{totaal}] {home} vs {away} ({kickoff.strftime("%Y-%m-%d")})', end=' ')

    try:
        slug, event = zoek_event(home, away, kickoff)
        tokens = get_tokens(event, home, away)

        if sorted(tokens.keys()) != ['away','draw','home']:
            msg = f'Incomplete tokens: {list(tokens.keys())}'
            print(f'⚠️  {msg}')
            fouten.append({'match_id': match_id, 'fout': msg})
            continue

        snap = {
            'match_id':   match_id, 'slug': slug,
            'season':     row['season'],
            'home_team':  home, 'away_team': away,
            'kickoff':    kickoff.isoformat(),
            'home_goals': row['home_goals'],
            'away_goals': row['away_goals'],
        }

        for label, token_id in tokens.items():
            df_hist = get_tijdreeks(token_id, kickoff)
            snap[f'prob_{label}'] = float(df_hist.iloc[0]['price']) if not df_hist.empty else None
            if not df_hist.empty:
                df_hist = df_hist.copy()
                df_hist[['match_id','slug','home_team','away_team','label']] = match_id, slug, home, away, label
                tijdreeksen.append(df_hist)
            time.sleep(0.2)

        # Verificatie som
        h = snap.get('prob_home') or 0
        d = snap.get('prob_draw') or 0
        a = snap.get('prob_away') or 0
        som = h + d + a
        flag = '✅' if abs(som-1.0) < 0.08 else '⚠️ som=%.3f' % som
        print(f'{flag}  H:{h:.2f} D:{d:.2f} A:{a:.2f}  {int(row["home_goals"])}-{int(row["away_goals"])}')
        snapshots.append(snap)

    except Exception as e:
        print(f'❌ {e}')
        fouten.append({'match_id': match_id, 'fout': str(e)})

    # Auto-save elke 10 wedstrijden
    if (i+1) % 10 == 0 and snapshots:
        df_tussen = pd.concat([df_snap_oud, pd.DataFrame(snapshots)], ignore_index=True)
        df_tussen.to_csv(SNAPSHOTS_PAD, index=False)
        if tijdreeksen:
            pd.concat([df_tr_oud] + tijdreeksen, ignore_index=True).to_csv(TIJDREEKS_PAD, index=False)
        print(f'  💾 Tussentijds opgeslagen na {i+1} pogingen')

print(f'\nKlaar: {len(snapshots)} OK, {len(fouten)} fouten')

Bestaande snapshots: 670 — sla deze over
Te ophalen: 392 wedstrijden
[1/392] Burnley vs Manchester City (2023-08-11) ❌ Geen event: Burnley vs Manchester City (2023-08-11)
[2/392] Arsenal vs Nottingham Forest (2023-08-12) ❌ Geen event: Arsenal vs Nottingham Forest (2023-08-12)
[3/392] Bournemouth vs West Ham United (2023-08-12) ❌ Geen event: Bournemouth vs West Ham United (2023-08-12)
[4/392] Sheffield United vs Crystal Palace (2023-08-12) ❌ Geen event: Sheffield United vs Crystal Palace (2023-08-12)
[5/392] Brighton & Hove Albion vs Luton Town (2023-08-12) ❌ Geen event: Brighton & Hove Albion vs Luton Town (2023-08-12)
[6/392] Everton vs Fulham (2023-08-12) ❌ Geen event: Everton vs Fulham (2023-08-12)
[7/392] Newcastle United vs Aston Villa (2023-08-12) ❌ Geen event: Newcastle United vs Aston Villa (2023-08-12)
[8/392] Brentford vs Tottenham Hotspur (2023-08-13) ❌ Geen event: Brentford vs Tottenham Hotspur (2023-08-13)
[9/392] Chelsea vs Liverpool (2023-08-13) ❌ Geen event: Chelsea vs 

C:\Users\semwi\AppData\Local\Temp\ipykernel_20540\4072540574.py:77: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_tussen = pd.concat([df_snap_oud, pd.DataFrame(snapshots)], ignore_index=True)


❌ Geen event: Bournemouth vs Tottenham Hotspur (2023-08-26)
[22/392] Brentford vs Crystal Palace (2023-08-26) ⚠️ som=0.000  H:0.00 D:0.00 A:0.00  1-1
[23/392] Arsenal vs Fulham (2023-08-26) ❌ Geen event: Arsenal vs Fulham (2023-08-26)
[24/392] Everton vs Wolverhampton (2023-08-26) ❌ Geen event: Everton vs Wolverhampton (2023-08-26)
[25/392] Manchester United vs Nottingham Forest (2023-08-26) ❌ Geen event: Manchester United vs Nottingham Forest (2023-08-26)
[26/392] Brighton & Hove Albion vs West Ham United (2023-08-26) ❌ Geen event: Brighton & Hove Albion vs West Ham United (2023-08-26)
[27/392] Burnley vs Aston Villa (2023-08-27) ❌ Geen event: Burnley vs Aston Villa (2023-08-27)
[28/392] Sheffield United vs Manchester City (2023-08-27) ❌ Geen event: Sheffield United vs Manchester City (2023-08-27)
[29/392] Newcastle United vs Liverpool (2023-08-27) ❌ Geen event: Newcastle United vs Liverpool (2023-08-27)
[30/392] Luton Town vs West Ham United (2023-09-01) ❌ Geen event: Luton Town vs W

C:\Users\semwi\AppData\Local\Temp\ipykernel_20540\4072540574.py:77: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_tussen = pd.concat([df_snap_oud, pd.DataFrame(snapshots)], ignore_index=True)


❌ Geen event: Sheffield United vs Everton (2023-09-02)
[32/392] Chelsea vs Nottingham Forest (2023-09-02) ⚠️ som=0.000  H:0.00 D:0.00 A:0.00  0-1
[33/392] Manchester City vs Fulham (2023-09-02) ⚠️ som=0.000  H:0.00 D:0.00 A:0.00  5-1
[34/392] Burnley vs Tottenham Hotspur (2023-09-02) ❌ Geen event: Burnley vs Tottenham Hotspur (2023-09-02)
[35/392] Brentford vs Bournemouth (2023-09-02) ❌ Geen event: Brentford vs Bournemouth (2023-09-02)
[36/392] Brighton & Hove Albion vs Newcastle United (2023-09-02) ❌ Geen event: Brighton & Hove Albion vs Newcastle United (2023-09-02)
[37/392] Liverpool vs Aston Villa (2023-09-03) ❌ Geen event: Liverpool vs Aston Villa (2023-09-03)
[38/392] Crystal Palace vs Wolverhampton (2023-09-03) ❌ Geen event: Crystal Palace vs Wolverhampton (2023-09-03)
[39/392] Arsenal vs Manchester United (2023-09-03) ❌ Geen event: Arsenal vs Manchester United (2023-09-03)
[40/392] Wolverhampton vs Liverpool (2023-09-16) ❌ Geen event: Wolverhampton vs Liverpool (2023-09-16)
  💾

C:\Users\semwi\AppData\Local\Temp\ipykernel_20540\4072540574.py:77: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_tussen = pd.concat([df_snap_oud, pd.DataFrame(snapshots)], ignore_index=True)


❌ Geen event: Aston Villa vs Crystal Palace (2023-09-16)
[42/392] West Ham United vs Manchester City (2023-09-16) ⚠️ som=0.000  H:0.00 D:0.00 A:0.00  1-3
[43/392] Manchester United vs Brighton & Hove Albion (2023-09-16) ❌ Geen event: Manchester United vs Brighton & Hove Albion (2023-09-16)
[44/392] Fulham vs Luton Town (2023-09-16) ❌ Geen event: Fulham vs Luton Town (2023-09-16)
[45/392] Tottenham Hotspur vs Sheffield United (2023-09-16) ❌ Geen event: Tottenham Hotspur vs Sheffield United (2023-09-16)
[46/392] Newcastle United vs Brentford (2023-09-16) ❌ Geen event: Newcastle United vs Brentford (2023-09-16)
[47/392] Bournemouth vs Chelsea (2023-09-17) ⚠️ som=0.000  H:0.00 D:0.00 A:0.00  0-0
[48/392] Everton vs Arsenal (2023-09-17) ❌ Geen event: Everton vs Arsenal (2023-09-17)
[49/392] Nottingham Forest vs Burnley (2023-09-18) ❌ Geen event: Nottingham Forest vs Burnley (2023-09-18)
[50/392] Manchester City vs Nottingham Forest (2023-09-23) ❌ Geen event: Manchester City vs Nottingham Fo

C:\Users\semwi\AppData\Local\Temp\ipykernel_20540\4072540574.py:77: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_tussen = pd.concat([df_snap_oud, pd.DataFrame(snapshots)], ignore_index=True)


❌ Geen event: Luton Town vs Wolverhampton (2023-09-23)
[52/392] Crystal Palace vs Fulham (2023-09-23) ❌ Geen event: Crystal Palace vs Fulham (2023-09-23)
[53/392] Brentford vs Everton (2023-09-23) ❌ Geen event: Brentford vs Everton (2023-09-23)
[54/392] Burnley vs Manchester United (2023-09-23) ❌ Geen event: Burnley vs Manchester United (2023-09-23)
[55/392] Arsenal vs Tottenham Hotspur (2023-09-24) ❌ Geen event: Arsenal vs Tottenham Hotspur (2023-09-24)
[56/392] Chelsea vs Aston Villa (2023-09-24) ❌ Geen event: Chelsea vs Aston Villa (2023-09-24)
[57/392] Liverpool vs West Ham United (2023-09-24) ❌ Geen event: Liverpool vs West Ham United (2023-09-24)
[58/392] Brighton & Hove Albion vs Bournemouth (2023-09-24) ❌ Geen event: Brighton & Hove Albion vs Bournemouth (2023-09-24)
[59/392] Sheffield United vs Newcastle United (2023-09-24) ❌ Geen event: Sheffield United vs Newcastle United (2023-09-24)
[60/392] Aston Villa vs Brighton & Hove Albion (2023-09-30) ❌ Geen event: Aston Villa vs Br

KeyboardInterrupt: 

In [16]:
# ── FINALE OPSLAG ────────────────────────────────────────────────────────────
if snapshots:
    df_snap_final = pd.concat([df_snap_oud, pd.DataFrame(snapshots)], ignore_index=True)
    df_snap_final.to_csv(SNAPSHOTS_PAD, index=False)
    print(f'Snapshots opgeslagen: {len(df_snap_final)} rijen')

    # Kwaliteitscheck
    check = df_snap_final[df_snap_final['prob_home'].notna()].copy()
    check['som'] = check['prob_home'] + check['prob_draw'] + check['prob_away']
    slecht = check[abs(check['som']-1.0) > 0.08]
    print(f'Verdachte rijen (som ver van 1): {len(slecht)}')
    if len(slecht):
        print(slecht[['home_team','away_team','prob_home','prob_draw','prob_away','som']])

if tijdreeksen:
    df_tr_final = pd.concat([df_tr_oud] + tijdreeksen, ignore_index=True)
    df_tr_final.to_csv(TIJDREEKS_PAD, index=False)
    print(f'Tijdreeksen opgeslagen: {len(df_tr_final)} rijen')

if fouten:
    pd.DataFrame(fouten).to_csv(FOUTEN_PAD, index=False)
    print(f'Fouten opgeslagen: {len(fouten)} in {FOUTEN_PAD}')

Snapshots opgeslagen: 669 rijen
Verdachte rijen (som ver van 1): 1
            home_team away_team  prob_home  prob_draw  prob_away    som
30  Nottingham Forest    Fulham      0.405        0.3        0.4  1.105
Tijdreeksen opgeslagen: 19112 rijen
Fouten opgeslagen: 12 in polymarket_fouten.csv


In [17]:
# ── UPDATE CEL: run dit periodiek voor nieuwe wedstrijden ────────────────────
nu              = pd.Timestamp.now()
een_maand_terug = nu - pd.DateOffset(months=1)

df_snap_huidig = pd.read_csv(SNAPSHOTS_PAD)
df_tr_huidig   = pd.read_csv(TIJDREEKS_PAD) if os.path.exists(TIJDREEKS_PAD) else pd.DataFrame()
al_opgehaald   = set(df_snap_huidig['match_id'])

# Wedstrijden in de afgelopen maand die nog niet opgehaald zijn
df_update = df_matches[
    (df_matches['timestamp'] >= een_maand_terug) &
    (~df_matches['match_id'].isin(al_opgehaald))
].reset_index(drop=True)

print(f'Update venster: {een_maand_terug.date()} → {nu.date()}')
print(f'Nieuwe wedstrijden te ophalen: {len(df_update)}')

# Zelfde lus als hierboven — copy-paste of zet in functie
# ... (voer de hoofdlus hierboven opnieuw uit met df_update i.p.v. te_ophalen)

Update venster: 2026-02-20 → 2026-03-20
Nieuwe wedstrijden te ophalen: 0
